# 🔬 Transformer-Based UNet for Rooftop PV Panel Segmentation

**Objective:** Transformer-based UNet with multihead attention architecture that surpasses baseline models performance on aerial photovoltaic panel segmentation across three benchmark datasets.

## 📊 Target Performance Goals

| Dataset | Baseline (DFDR-NLNet) mIoU | Target mIoU |
|---------|---------------------------|-------------|
| PV01    | 91.48%                    | > 92.0%     |
| PVP     | 83.39%                    | > 84.0%     |
| BDAPPV  | 66.14%                    | > 67.0%     |

## 📋 goals
- ✅ Transformer encoder with multi-head self-attention
- ✅ UNet-style decoder with skip connections for edge preservation
- ✅ Training on 3 datasets: PVP, BDAPPV, PV01
- ✅ Comprehensive metrics: mIoU, F1-Score, Inference Time, Parameters, FLOPs
- ✅ Statistical significance testing (paired t-test)
- ✅ Data augmentation for robustness



In [1]:
# ============================================================================
# ENVIRONMENT SETUP AND GPU VERIFICATION
# ============================================================================
# This cell checks the runtime environment and verifies GPU availability,
# which is critical for efficient model training.
# ============================================================================

import sys
import torch
import platform

print("=" * 70)
print("ENVIRONMENT CONFIGURATION")
print("=" * 70)

# Check Python version
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")

# Check PyTorch version
print(f"\nPyTorch Version: {torch.__version__}")

# Verify GPU availability and specifications
if torch.cuda.is_available():
    print(f"\n✅ GPU is AVAILABLE")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("\n❌ WARNING: GPU not available. Training will be very slow!")
    print("Please change runtime: Runtime → Change runtime type → GPU")

print("=" * 70)

ENVIRONMENT CONFIGURATION
Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35

PyTorch Version: 2.9.0+cu126

✅ GPU is AVAILABLE
GPU Device: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB
CUDA Version: 12.6


## 📦 Installation & Dependencies

Installing required libraries for:
- Deep learning framework (PyTorch)
- Image augmentation (Albumentations)
- Segmentation metrics (segmentation-models-pytorch)
- Model analysis (thop for FLOPs calculation)
- Visualization tools

In [2]:
# ============================================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================================
# Installing all necessary packages for model development, training,
# and evaluation. This includes libraries for data handling, augmentation,
# model architecture, metrics calculation, and FLOPs computation.
# ============================================================================

print("Installing required packages..\n")

!pip install -q albumentations==1.3.1
!pip install -q segmentation-models-pytorch==0.3.3
!pip install -q thop  # For FLOPs calculation
!pip install -q timm  # Transformer backbones
!pip install -q scikit-learn
!pip install -q matplotlib seaborn
!pip install -q tqdm

print("\n✅ All packages installed successfully!")

# Verify key installations
import albumentations as A
import segmentation_models_pytorch as smp
from thop import profile
import timm

print(f"\nAlbumentations version: {A.__version__}")
print(f"Segmentation Models PyTorch version: {smp.__version__}")
print(f"TIMM version: {timm.__version__}")

Installing required packages..

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.9 MB/s eta 0:00:00

✅ All packages installed successfully!

Albumentations version: 1.3.1
Segmentation Models PyTorch version: 0.3.3
TIMM version: 0.9.2


## 💾 Mount Google Drive

Mounting Google Drive to:
- Save trained models and checkpoints

- Save evaluation results and visualizations
- Ensure work persistence across sessions

In [3]:
# ============================================================================
# MOUNT GOOGLE DRIVE
# ============================================================================
# Mount Google Drive to persist datasets, models, and results.
# Creates a dedicated project directory for organized file management.
# ============================================================================

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory structure
PROJECT_DIR = '/content/drive/MyDrive/TransformerUNet_PV_Segmentation'
DATASETS_DIR = os.path.join(PROJECT_DIR, 'datasets')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')

# Create directories if they don't exist
os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("✅ Google Drive mounted successfully!")
print(f"\n📁 Project Directory: {PROJECT_DIR}")
print(f"📁 Datasets Directory: {DATASETS_DIR}")
print(f"📁 Models Directory: {MODELS_DIR}")
print(f"📁 Results Directory: {RESULTS_DIR}")

Mounted at /content/drive
✅ Google Drive mounted successfully!

📁 Project Directory: /content/drive/MyDrive/TransformerUNet_PV_Segmentation
📁 Datasets Directory: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/datasets
📁 Models Directory: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/models
📁 Results Directory: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/results


## 📚 Import Required Libraries

Importing all necessary modules for:
- Data handling and preprocessing
- Model architecture development
- Training and evaluation
- Metrics calculation and visualization

In [4]:
# ============================================================================
# IMPORT REQUIRED LIBRARIES (WITH ERROR HANDLING)
# ============================================================================
# Comprehensive imports with fallback installation for missing packages
# ============================================================================

# Standard library imports
import os
import sys
import time
import random
import warnings
from pathlib import Path
from typing import Tuple, List, Dict, Optional

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Data handling and processing
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Deep learning framework
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Image augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Model architectures - with fallback installation
try:
    import timm
    import segmentation_models_pytorch as smp
except ImportError:
    print("Installing missing model packages...")
    !pip install -q segmentation-models-pytorch timm
    import timm
    import segmentation_models_pytorch as smp

# Metrics and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from scipy import stats

# FLOPs calculation
try:
    from thop import profile, clever_format
except ImportError:
    !pip install -q thop
    from thop import profile, clever_format

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 70)
print("✅ All libraries imported successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"TIMM: {timm.__version__}")
print(f"SMP: {smp.__version__}")
print(f"Albumentations: {A.__version__}")
print("=" * 70)

✅ All libraries imported successfully!
PyTorch: 2.9.0+cu126
TIMM: 0.9.2
SMP: 0.3.3
Albumentations: 1.3.1


## ⚙️ Configuration & Hyperparameters

Setting up all training configurations including:
- Image dimensions and batch sizes
- Learning rates and optimization parameters
- Transformer model architecture specifications
- Training epochs and early stopping criteria
- Random seeds for reproducibility
- Loss functions tailored for edge-preserving segmentation

In [5]:
import os
import sys
import time
import random
import warnings
from pathlib import Path
from typing import Tuple, List, Dict, Optional

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Data handling and processing
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Deep learning framework
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Image augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Model architectures - with fallback installation
try:
    import timm
    import segmentation_models_pytorch as smp
except ImportError:
    print("Installing missing model packages...")
    !pip install -q segmentation-models-pytorch timm
    import timm
    import segmentation_models_pytorch as smp

# Metrics and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from scipy import stats

# FLOPs calculation
try:
    from thop import profile, clever_format
except ImportError:
    !pip install -q thop
    from thop import profile, clever_format

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class Config:
    """
    Central configuration class containing all hyperparameters and settings
    for Transformer-based UNet training, evaluation, and deployment.

    Designed to surpass baseline:
    - PV01: 91.48% → Target: >92%
    - PVP: 83.39% → Target: >84%
    - BDAPPV: 66.14% → Target: >67%
    """

    # ======================== RANDOM SEEDS ========================
    SEED = 42  # For reproducibility across all experiments

    # ======================== DEVICE SETUP ========================
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    NUM_GPUS = torch.cuda.device_count()

    # ======================== DATA PARAMETERS ========================
    IMG_SIZE = 512  # 512x512 for better edge detail preservation
    BATCH_SIZE = 8  # Optimized for A100 80GB GPU
    NUM_WORKERS = 4  # DataLoader parallel workers
    PIN_MEMORY = True  # Faster data transfer to GPU

    # Dataset split ratios
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    # ======================== MODEL PARAMETERS ========================
    NUM_CLASSES = 1  # Binary segmentation: background (0) vs PV panel (1)

    # Transformer encoder backbone
    ENCODER_NAME = 'mit_b3'  # Mix Transformer B3 - balanced performance/efficiency
    ENCODER_WEIGHTS = 'imagenet'  # ImageNet pre-trained weights

    # Alternative backbones to experiment with:
    # 'swin_base_patch4_window7_224' - Swin Transformer
    # 'vit_base_patch16_224' - Vision Transformer

    # Attention mechanism parameters
    NUM_ATTENTION_HEADS = 8
    DROPOUT_RATE = 0.1

    # ======================== TRAINING PARAMETERS ========================
    EPOCHS = 150  # Maximum training epochs
    LEARNING_RATE = 1e-4  # Initial learning rate
    WEIGHT_DECAY = 1e-5  # L2 regularization

    # Learning rate scheduler (cosine annealing for smooth convergence)
    SCHEDULER_TYPE = 'cosine'
    MIN_LR = 1e-6  # Minimum learning rate
    WARMUP_EPOCHS = 5  # Warmup epochs for stable training start

    # Early stopping to prevent overfitting
    PATIENCE = 20  # Stop if no improvement for 20 epochs
    MIN_DELTA = 0.001  # Minimum improvement threshold

    # Mixed precision training
    USE_AMP = True  # Automatic Mixed Precision (FP16)

    # Gradient clipping for stable training
    CLIP_GRAD_NORM = 1.0

    # ======================== LOSS FUNCTION ========================
    # Hybrid loss for better edge detection and segmentation accuracy
    LOSS_TYPE = 'dice_focal'  # Combined Dice + Focal Loss

    # Loss weights (tuned for PV panel segmentation)
    DICE_WEIGHT = 0.6  # Emphasis on overlap
    FOCAL_WEIGHT = 0.4  # Handle class imbalance
    FOCAL_ALPHA = 0.25  # Focal loss alpha
    FOCAL_GAMMA = 2.0  # Focal loss gamma

    # ======================== DATA AUGMENTATION ========================
    # Augmentation probabilities
    AUG_ROTATE_PROB = 0.5
    AUG_FLIP_PROB = 0.5
    AUG_BRIGHTNESS_PROB = 0.3
    AUG_CONTRAST_PROB = 0.3

    # ======================== PATHS ========================
    PROJECT_DIR = '/content/drive/MyDrive/TransformerUNet_PV_Segmentation'
    DATASETS_DIR = os.path.join(PROJECT_DIR, 'datasets')
    MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
    RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
    CHECKPOINTS_DIR = os.path.join(MODELS_DIR, 'checkpoints')

    # Create checkpoint directory
    os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

    # ======================== EVALUATION METRICS ========================
    METRICS = ['mIoU', 'F1', 'Precision', 'Recall', 'Accuracy']

    # ======================== INFERENCE & BENCHMARKING ========================
    INFERENCE_BATCH_SIZE = 1  # For accurate timing measurement
    NUM_INFERENCE_RUNS = 100  # Average over multiple runs for stable timing
    CALCULATE_FLOPS = True  # Calculate FLOPs during evaluation

    # ======================== VISUALIZATION ========================
    SAVE_PREDICTIONS = True  # Save prediction visualizations
    NUM_VISUAL_SAMPLES = 10  # Number of samples to visualize per dataset


def set_seed(seed: int = Config.SEED):
    """
    Set random seeds for reproducibility across all libraries.
    Essential for consistent experimental results.

    Args:
        seed (int): Random seed value
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensure deterministic behavior (may slightly reduce performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Set environment variable for additional reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)


# Apply random seed
set_seed()

# Display configuration summary
print("=" * 70)
print("TRANSFORMER-BASED UNET CONFIGURATION")
print("=" * 70)
print(f"Device: {Config.DEVICE}")
print(f"Number of GPUs: {Config.NUM_GPUS}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nImage Size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"Batch Size: {Config.BATCH_SIZE}")
print(f"Epochs: {Config.EPOCHS}")
print(f"Learning Rate: {Config.LEARNING_RATE}")
print(f"Encoder Backbone: {Config.ENCODER_NAME}")
print(f"Loss Function: {Config.LOSS_TYPE}")
print(f"Mixed Precision: {Config.USE_AMP}")
print(f"\nDataset Split: Train {Config.TRAIN_RATIO:.0%} | Val {Config.VAL_RATIO:.0%} | Test {Config.TEST_RATIO:.0%}")
print(f"Random Seed: {Config.SEED}")
print("=" * 70)

TRANSFORMER-BASED UNET CONFIGURATION
Device: cuda
Number of GPUs: 1
GPU Name: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB

Image Size: 512x512
Batch Size: 8
Epochs: 150
Learning Rate: 0.0001
Encoder Backbone: mit_b3
Loss Function: dice_focal
Mixed Precision: True

Dataset Split: Train 70% | Val 15% | Test 15%
Random Seed: 42


## 📂 Dataset Organization & Utility Functions

Creating helper functions for:
- Dataset verification and structure validation
- File counting and organization
- Path management across three datasets (PVP, BDAPPV, PV01)
- Data integrity checks


In [6]:
# ============================================================================
# ENHANCED DATASET SCANNER - SHOWS ALL SUBFOLDERS
# ============================================================================

def scan_dataset_detailed(dataset_path: str, dataset_name: str):
    """
    Detailed scanner that explicitly shows progress for each subfolder.
    Handles all three dataset structures perfectly.
    """
    print(f"\n{'='*70}")
    print(f"Scanning {dataset_name} Dataset")
    print(f"{'='*70}")

    if not os.path.exists(dataset_path):
        print(f"❌ Not found: {dataset_path}")
        return {'image_files': [], 'mask_files': [], 'num_images': 0, 'num_masks': 0}

    image_files = []
    mask_files = []

    # Track which folders we've scanned
    scanned_folders = set()

    # Walk through ALL directories and subdirectories
    for root, dirs, files in os.walk(dataset_path):
        if not files:  # Skip empty folders
            continue

        # Show which folder we're scanning
        relative_path = root.replace(dataset_path, '').strip('/')
        if relative_path and relative_path not in scanned_folders:
            print(f"  📂 Scanning: {relative_path}/")
            scanned_folders.add(relative_path)

        for file in files:
            # Check all image formats (including .bmp for PV01)
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')):
                filepath = os.path.join(root, file)
                filename_lower = file.lower()

                # Determine if it's a mask or image
                is_mask = False

                # Method 1: Check filename for label/mask keywords
                if any(keyword in filename_lower for keyword in ['_label', 'label_', '_mask', 'mask_', '_gt']):
                    is_mask = True
                # Method 2: Check if parent folder name suggests masks
                elif any(keyword in root.lower() for keyword in ['/mask', '/label', '/gt', '/annotation']):
                    is_mask = True

                if is_mask:
                    mask_files.append(filepath)
                else:
                    image_files.append(filepath)

    # Remove duplicates
    image_files = list(set(image_files))
    mask_files = list(set(mask_files))

    print(f"\n  ✅ Total Images: {len(image_files)}")
    print(f"  ✅ Total Masks: {len(mask_files)}")

    return {
        'image_files': sorted(image_files),
        'mask_files': sorted(mask_files),
        'num_images': len(image_files),
        'num_masks': len(mask_files)
    }


# Scan all three datasets
print("=" * 70)
print("🔍 COMPREHENSIVE DATASET SCAN")
print("=" * 70)

all_dataset_info = {}
datasets = {
    'PVP': os.path.join(Config.DATASETS_DIR, 'PVP'),
    'BDAPPV': os.path.join(Config.DATASETS_DIR, 'BDAPPV'),
    'PV01': os.path.join(Config.DATASETS_DIR, 'PV01')
}

for name, path in datasets.items():
    all_dataset_info[name] = scan_dataset_detailed(path, name)

# Final Summary
print(f"\n{'='*70}")
print("📊 FINAL DATASET SUMMARY")
print(f"{'='*70}")

total_images = 0
total_masks = 0

for name in ['PVP', 'BDAPPV', 'PV01']:  # Ordered display
    info = all_dataset_info[name]
    total_images += info['num_images']
    total_masks += info['num_masks']

    status = '✅ READY' if info['num_images'] > 0 and info['num_masks'] > 0 else '⚠️ INCOMPLETE'

    print(f"\n{name}: {status}")
    print(f"  • Images: {info['num_images']:,}")
    print(f"  • Masks: {info['num_masks']:,}")
    if info['num_images'] != info['num_masks']:
        print(f"  • Note: Different counts (normal for some datasets)")

print(f"\n{'='*70}")
print(f"🎯 GRAND TOTAL")
print(f"{'='*70}")
print(f"  Total Images: {total_images:,}")
print(f"  Total Masks: {total_masks:,}")
print(f"{'='*70}")

print("\n✅ All datasets loaded and ready for training!")

🔍 COMPREHENSIVE DATASET SCAN

Scanning PVP Dataset
  📂 Scanning: PVP/
  📂 Scanning: PVP/images/
  📂 Scanning: PVP/labels/

  ✅ Total Images: 4640
  ✅ Total Masks: 4640

Scanning BDAPPV Dataset
  📂 Scanning: ign/
  📂 Scanning: ign/mask/
  📂 Scanning: ign/img/

  ✅ Total Images: 17325
  ✅ Total Masks: 7685

Scanning PV01 Dataset
  📂 Scanning: PV01/
  📂 Scanning: PV01/PV01_Rooftop_FlatConcrete/
  📂 Scanning: PV01/PV01_Rooftop_Brick/
  📂 Scanning: PV01/PV01_Rooftop_SteelTile/

  ✅ Total Images: 645
  ✅ Total Masks: 645

📊 FINAL DATASET SUMMARY

PVP: ✅ READY
  • Images: 4,640
  • Masks: 4,640

BDAPPV: ✅ READY
  • Images: 17,325
  • Masks: 7,685
  • Note: Different counts (normal for some datasets)

PV01: ✅ READY
  • Images: 645
  • Masks: 645

🎯 GRAND TOTAL
  Total Images: 22,610
  Total Masks: 12,970

✅ All datasets loaded and ready for training!


## 🎨 Data Augmentation Pipeline

NOTE: This was done as with experimentation, I got better results

Creating robust augmentation strategies to improve model generalization:
- **Geometric transforms**: Rotation, flipping (simulating different drone angles)
- **Color adjustments**: Brightness, contrast (handling varying lighting conditions)
- **Normalization**: Using ImageNet statistics for pre-trained encoder compatibility

These augmentations help the model handle real-world drone imagery variations.

In [7]:
# ============================================================================
# DATA AUGMENTATION PIPELINE
# ============================================================================
# Comprehensive augmentation for aerial PV panel imagery to improve model
# robustness across different lighting, angles, and environmental conditions.
# ============================================================================

def get_training_augmentation(img_size: int = Config.IMG_SIZE):
    """
    Training augmentation pipeline with geometric and color transforms.

    Args:
        img_size (int): Target image size

    Returns:
        albumentations.Compose: Augmentation pipeline
    """
    train_transform = A.Compose([
        # Resize to target size
        A.Resize(img_size, img_size, always_apply=True),

        # Geometric transformations (drone viewing angle variations)
        A.HorizontalFlip(p=Config.AUG_FLIP_PROB),
        A.VerticalFlip(p=Config.AUG_FLIP_PROB),
        A.RandomRotate90(p=0.5),

        # Small rotations for realistic drone angle variations
        A.Rotate(limit=15, p=0.3, border_mode=cv2.BORDER_CONSTANT, value=0),

        # Color/lighting augmentations (time of day, weather variations)
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=Config.AUG_BRIGHTNESS_PROB
        ),

        # Additional photometric augmentations
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=20,
            val_shift_limit=10,
            p=0.2
        ),

        # Normalize using ImageNet statistics (required for pre-trained encoders)
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            always_apply=True
        ),

        # Convert to PyTorch tensor
        ToTensorV2(always_apply=True)
    ])

    return train_transform


def get_validation_augmentation(img_size: int = Config.IMG_SIZE):
    """
    Validation/test augmentation (only resize and normalize, no random transforms).

    Args:
        img_size (int): Target image size

    Returns:
        albumentations.Compose: Augmentation pipeline
    """
    val_transform = A.Compose([
        A.Resize(img_size, img_size, always_apply=True),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            always_apply=True
        ),
        ToTensorV2(always_apply=True)
    ])

    return val_transform


# Test augmentation pipeline
print("=" * 70)
print("AUGMENTATION PIPELINES CREATED")
print("=" * 70)

train_aug = get_training_augmentation()
val_aug = get_validation_augmentation()

print("\n✅ Training Augmentation:")
print(f"  - Resize to {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"  - Horizontal/Vertical Flip (p={Config.AUG_FLIP_PROB})")
print(f"  - Random Rotation ±15° (p=0.3)")
print(f"  - Brightness/Contrast (p={Config.AUG_BRIGHTNESS_PROB})")
print(f"  - HSV adjustments (p=0.2)")
print(f"  - ImageNet Normalization")

print("\n✅ Validation Augmentation:")
print(f"  - Resize to {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"  - ImageNet Normalization only")

print("=" * 70)

AUGMENTATION PIPELINES CREATED

✅ Training Augmentation:
  - Resize to 512x512
  - Horizontal/Vertical Flip (p=0.5)
  - Random Rotation ±15° (p=0.3)
  - Brightness/Contrast (p=0.3)
  - HSV adjustments (p=0.2)
  - ImageNet Normalization

✅ Validation Augmentation:
  - Resize to 512x512
  - ImageNet Normalization only


## 📦 Custom PyTorch Dataset Class

Creating a flexible dataset class that:
- Loads images and masks from all three datasets (PVP, BDAPPV, PV01)
- Handles different file formats (.png, .bmp)
- Applies augmentation transforms
- Properly pairs images with their corresponding masks
- Converts masks to binary format (background=0, PV panel=1)

In [8]:
# ============================================================================
# CUSTOM PYTORCH DATASET CLASS
# ============================================================================
# Flexible dataset class handling all three PV panel datasets with different
# formats and structures. Includes proper image-mask pairing and preprocessing.
# ============================================================================

class PVPanelDataset(Dataset):
    """
    Custom Dataset for PV Panel Segmentation across multiple datasets.

    Handles:
    - PVP: .png images with separate images/ and labels/ folders
    - BDAPPV: .png images with img/ and mask/ subfolders
    - PV01: .bmp images with _label suffix in filenames
    """

    def __init__(
        self,
        image_files: List[str],
        mask_files: List[str],
        transform=None,
        dataset_name: str = "Unknown"
    ):
        """
        Initialize dataset.

        Args:
            image_files: List of image file paths
            mask_files: List of mask file paths
            transform: Albumentations transform pipeline
            dataset_name: Name of the dataset for logging
        """
        self.image_files = image_files
        self.mask_files = mask_files
        self.transform = transform
        self.dataset_name = dataset_name

        # Create image-mask pairs
        self.pairs = self._create_pairs()

        print(f"✅ {dataset_name} Dataset initialized: {len(self.pairs)} image-mask pairs")

    def _create_pairs(self) -> List[Tuple[str, str]]:
        """
        Create image-mask pairs by matching filenames.

        Returns:
            List of (image_path, mask_path) tuples
        """
        pairs = []

        # Create a mapping of base names to mask files for quick lookup
        mask_dict = {}
        for mask_path in self.mask_files:
            mask_name = os.path.basename(mask_path)
            # Remove mask-specific suffixes to get base name
            base_name = mask_name.replace('_label', '').replace('_mask', '').replace('_gt', '')
            mask_dict[base_name] = mask_path

        # Match each image to its mask
        for img_path in self.image_files:
            img_name = os.path.basename(img_path)

            # Try to find corresponding mask
            if img_name in mask_dict:
                pairs.append((img_path, mask_dict[img_name]))
            else:
                # Try with different extensions
                base_name = os.path.splitext(img_name)[0]
                for ext in ['.png', '.bmp', '.jpg', '.tif']:
                    candidate = base_name + ext
                    if candidate in mask_dict:
                        pairs.append((img_path, mask_dict[candidate]))
                        break

        return pairs

    def __len__(self) -> int:
        """Return dataset size."""
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Get a single image-mask pair.

        Args:
            idx: Index

        Returns:
            Dictionary with 'image' and 'mask' tensors
        """
        img_path, mask_path = self.pairs[idx]

        # Load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Binarize mask (ensure it's 0 or 1)
        # Assume any non-zero value is PV panel (class 1)
        mask = (mask > 0).astype(np.uint8)

        # Apply augmentations
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        else:
            # Manual tensor conversion if no transform
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).float()

        # Add channel dimension if missing: [H, W] → [1, H, W]
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        # Ensure binary mask [0, 1]
        mask = (mask > 0.5).float()

        return {
            'image': image,
            'mask': mask,
            'image_path': img_path,
            'mask_path': mask_path
        }


# Test dataset creation
print("=" * 70)
print("TESTING DATASET CLASS")
print("=" * 70)

# Create a small test dataset with PVP data
test_dataset = PVPanelDataset(
    image_files=all_dataset_info['PVP']['image_files'][:100],  # First 100 images
    mask_files=all_dataset_info['PVP']['mask_files'][:100],
    transform=get_validation_augmentation(),
    dataset_name="PVP_Test"
)

# Test loading a sample
print(f"\n📊 Testing data loading...")
sample = test_dataset[0]
print(f"  Image shape: {sample['image'].shape}")
print(f"  Mask shape: {sample['mask'].shape}")
print(f"  Mask unique values: {torch.unique(sample['mask']).tolist()}")
print(f"  Image dtype: {sample['image'].dtype}")
print(f"  Mask dtype: {sample['mask'].dtype}")

print("\n✅ Dataset class working perfectly!")
print("=" * 70)

TESTING DATASET CLASS
✅ PVP_Test Dataset initialized: 100 image-mask pairs

📊 Testing data loading...
  Image shape: torch.Size([3, 512, 512])
  Mask shape: torch.Size([1, 512, 512])
  Mask unique values: [0.0, 1.0]
  Image dtype: torch.float32
  Mask dtype: torch.float32

✅ Dataset class working perfectly!


## 🔀 Train/Validation/Test Split

Creating proper data splits for all three datasets:
- **70% Training** - For model learning
- **15% Validation** - For hyperparameter tuning and early stopping
- **15% Testing** - For final performance evaluation

Each dataset (PVP, BDAPPV, PV01) will be split independently to ensure fair evaluation across all benchmarks.

In [9]:
# ============================================================================
# CREATE TRAIN/VALIDATION/TEST SPLITS
# ============================================================================
# Split each dataset (PVP, BDAPPV, PV01) into train/val/test sets
# using stratified random sampling with fixed random seed for reproducibility
# ============================================================================

def create_dataset_splits(
    image_files: List[str],
    mask_files: List[str],
    dataset_name: str,
    train_ratio: float = Config.TRAIN_RATIO,
    val_ratio: float = Config.VAL_RATIO,
    test_ratio: float = Config.TEST_RATIO,
    seed: int = Config.SEED
) -> Dict[str, Dict[str, List[str]]]:
    """
    Split dataset into train/val/test sets.

    Args:
        image_files: List of image paths
        mask_files: List of mask paths
        dataset_name: Name for logging
        train_ratio: Training set ratio
        val_ratio: Validation set ratio
        test_ratio: Test set ratio
        seed: Random seed

    Returns:
        Dictionary with train/val/test splits
    """
    print(f"\n{'='*70}")
    print(f"Splitting {dataset_name} Dataset")
    print(f"{'='*70}")

    # Create dataset instance to get proper image-mask pairs
    temp_dataset = PVPanelDataset(
        image_files=image_files,
        mask_files=mask_files,
        transform=None,
        dataset_name=dataset_name
    )

    # Get all pairs
    all_pairs = temp_dataset.pairs
    total_samples = len(all_pairs)

    print(f"Total samples: {total_samples}")

    # Calculate split sizes
    train_size = int(total_samples * train_ratio)
    val_size = int(total_samples * val_ratio)
    test_size = total_samples - train_size - val_size

    print(f"  Train: {train_size} ({train_ratio*100:.0f}%)")
    print(f"  Val: {val_size} ({val_ratio*100:.0f}%)")
    print(f"  Test: {test_size} ({test_ratio*100:.0f}%)")

    # Shuffle and split
    np.random.seed(seed)
    indices = np.random.permutation(total_samples)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size+val_size]
    test_indices = indices[train_size+val_size:]

    # Extract paths for each split
    train_pairs = [all_pairs[i] for i in train_indices]
    val_pairs = [all_pairs[i] for i in val_indices]
    test_pairs = [all_pairs[i] for i in test_indices]

    splits = {
        'train': {
            'image_files': [p[0] for p in train_pairs],
            'mask_files': [p[1] for p in train_pairs]
        },
        'val': {
            'image_files': [p[0] for p in val_pairs],
            'mask_files': [p[1] for p in val_pairs]
        },
        'test': {
            'image_files': [p[0] for p in test_pairs],
            'mask_files': [p[1] for p in test_pairs]
        }
    }

    print(f"✅ Split complete!")

    return splits


# Create splits for all three datasets
print("=" * 70)
print("CREATING DATASET SPLITS")
print("=" * 70)

dataset_splits = {}

for dataset_name in ['PVP', 'BDAPPV', 'PV01']:
    info = all_dataset_info[dataset_name]

    splits = create_dataset_splits(
        image_files=info['image_files'],
        mask_files=info['mask_files'],
        dataset_name=dataset_name
    )

    dataset_splits[dataset_name] = splits

# Summary
print(f"\n{'='*70}")
print("SPLIT SUMMARY")
print(f"{'='*70}")

for dataset_name, splits in dataset_splits.items():
    print(f"\n{dataset_name}:")
    print(f"  Train: {len(splits['train']['image_files'])} samples")
    print(f"  Val: {len(splits['val']['image_files'])} samples")
    print(f"  Test: {len(splits['test']['image_files'])} samples")

print(f"\n{'='*70}")
print("✅ All datasets split successfully!")
print("=" * 70)

CREATING DATASET SPLITS

Splitting PVP Dataset
✅ PVP Dataset initialized: 4640 image-mask pairs
Total samples: 4640
  Train: 3248 (70%)
  Val: 696 (15%)
  Test: 696 (15%)
✅ Split complete!

Splitting BDAPPV Dataset
✅ BDAPPV Dataset initialized: 7685 image-mask pairs
Total samples: 7685
  Train: 5379 (70%)
  Val: 1152 (15%)
  Test: 1154 (15%)
✅ Split complete!

Splitting PV01 Dataset
✅ PV01 Dataset initialized: 645 image-mask pairs
Total samples: 645
  Train: 451 (70%)
  Val: 96 (15%)
  Test: 98 (15%)
✅ Split complete!

SPLIT SUMMARY

PVP:
  Train: 3248 samples
  Val: 696 samples
  Test: 696 samples

BDAPPV:
  Train: 5379 samples
  Val: 1152 samples
  Test: 1154 samples

PV01:
  Train: 451 samples
  Val: 96 samples
  Test: 98 samples

✅ All datasets split successfully!


## 🔄 DataLoader Creation

Creating PyTorch DataLoaders for efficient batch processing:
- **Training DataLoader**: With shuffling and augmentation
- **Validation DataLoader**: Without shuffling, deterministic
- **Test DataLoader**: For final evaluation

We'll create DataLoaders for each dataset separately to enable per-dataset evaluation.

In [10]:
# ============================================================================
# CREATE PYTORCH DATALOADERS
# ============================================================================
# Create efficient DataLoaders for training, validation, and testing
# across all three datasets (PVP, BDAPPV, PV01).
# ============================================================================

def create_dataloaders(
    dataset_name: str,
    splits: Dict[str, Dict[str, List[str]]],
    batch_size: int = Config.BATCH_SIZE,
    num_workers: int = Config.NUM_WORKERS
) -> Dict[str, DataLoader]:
    """
    Create DataLoaders for train/val/test splits.

    Args:
        dataset_name: Name of the dataset
        splits: Dictionary with train/val/test splits
        batch_size: Batch size for training
        num_workers: Number of worker processes

    Returns:
        Dictionary with train/val/test DataLoaders
    """
    print(f"\n{'='*70}")
    print(f"Creating DataLoaders for {dataset_name}")
    print(f"{'='*70}")

    # Training dataset with augmentation
    train_dataset = PVPanelDataset(
        image_files=splits['train']['image_files'],
        mask_files=splits['train']['mask_files'],
        transform=get_training_augmentation(),
        dataset_name=f"{dataset_name}_Train"
    )

    # Validation dataset without augmentation
    val_dataset = PVPanelDataset(
        image_files=splits['val']['image_files'],
        mask_files=splits['val']['mask_files'],
        transform=get_validation_augmentation(),
        dataset_name=f"{dataset_name}_Val"
    )

    # Test dataset without augmentation
    test_dataset = PVPanelDataset(
        image_files=splits['test']['image_files'],
        mask_files=splits['test']['mask_files'],
        transform=get_validation_augmentation(),
        dataset_name=f"{dataset_name}_Test"
    )

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,  # Shuffle for training
        num_workers=num_workers,
        pin_memory=Config.PIN_MEMORY,
        drop_last=True  # Drop incomplete batches for stable training
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,  # Don't shuffle validation
        num_workers=num_workers,
        pin_memory=Config.PIN_MEMORY
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,  # Don't shuffle test
        num_workers=num_workers,
        pin_memory=Config.PIN_MEMORY
    )

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")

    return {
        'train': train_loader,
        'val': val_loader,
        'test': test_loader
    }


# Create DataLoaders for all datasets
print("=" * 70)
print("CREATING DATALOADERS FOR ALL DATASETS")
print("=" * 70)

all_dataloaders = {}

for dataset_name in ['PVP', 'BDAPPV', 'PV01']:
    dataloaders = create_dataloaders(
        dataset_name=dataset_name,
        splits=dataset_splits[dataset_name]
    )
    all_dataloaders[dataset_name] = dataloaders

# Summary
print(f"\n{'='*70}")
print("DATALOADER SUMMARY")
print(f"{'='*70}")

for dataset_name, loaders in all_dataloaders.items():
    print(f"\n{dataset_name}:")
    print(f"  Train: {len(loaders['train'])} batches")
    print(f"  Val: {len(loaders['val'])} batches")
    print(f"  Test: {len(loaders['test'])} batches")

print(f"\n{'='*70}")
print("✅ All DataLoaders created successfully!")
print(f"Batch size: {Config.BATCH_SIZE}")
print("=" * 70)

CREATING DATALOADERS FOR ALL DATASETS

Creating DataLoaders for PVP
✅ PVP_Train Dataset initialized: 3248 image-mask pairs
✅ PVP_Val Dataset initialized: 696 image-mask pairs
✅ PVP_Test Dataset initialized: 696 image-mask pairs
  Train batches: 406
  Val batches: 87
  Test batches: 87

Creating DataLoaders for BDAPPV
✅ BDAPPV_Train Dataset initialized: 5379 image-mask pairs
✅ BDAPPV_Val Dataset initialized: 1152 image-mask pairs
✅ BDAPPV_Test Dataset initialized: 1154 image-mask pairs
  Train batches: 672
  Val batches: 144
  Test batches: 145

Creating DataLoaders for PV01
✅ PV01_Train Dataset initialized: 451 image-mask pairs
✅ PV01_Val Dataset initialized: 96 image-mask pairs
✅ PV01_Test Dataset initialized: 98 image-mask pairs
  Train batches: 56
  Val batches: 12
  Test batches: 13

DATALOADER SUMMARY

PVP:
  Train: 406 batches
  Val: 87 batches
  Test: 87 batches

BDAPPV:
  Train: 672 batches
  Val: 144 batches
  Test: 145 batches

PV01:
  Train: 56 batches
  Val: 12 batches
  Te

## 🏗️ Mix Transformer (MiT-B4) UNet Model Architecture

Building our segmentation model optimized for aerial PV panel detection:

### Encoder (Mix Transformer - MiT-B4):
- **Mix Transformer B4** backbone with hierarchical 4-stage feature extraction
- **Multi-head self-attention** at multiple scales for global context
- **Overlapping patch embedding** for better local continuity
- **No positional encoding** = adapts to varying image resolutions
- **Pre-trained on ImageNet-1K** for superior feature extraction
- **~62M parameters** - optimal balance of capacity and efficiency

### Decoder (UNet-style with Attention):
- **Skip connections** from all encoder stages for multi-scale edge preservation
- **Progressive upsampling** (16x → 8x → 4x → 2x → 1x)
- **scSE Attention** (Spatial & Channel Squeeze-Excitation) at each decoder block
- **Feature fusion** combining low-level edges with high-level semantics
- **Lightweight design** (~5M parameters) for computational efficiency


### Key Advantages:
- **🌍 Global context**: Transformer self-attention captures long-range dependencies
- **📏 Edge precision**: Hierarchical features + scSE attention preserve panel boundaries
- **⚡ Efficiency**: ~67M total params, optimized for real-time drone inference
- **🎯 Multi-resolution**: Handles varying dataset qualities (PVP, BDAPPV, PV01)
- **🔬 Proven**: Specifically validated on photovoltaic panel segmentation tasks

In [11]:
# ============================================================================
# MIX TRANSFORMER (MiT-B4) UNET MODEL
# ============================================================================
# architecture combining:
# - Mix Transformer (MiT-B4) encoder for global context
# - UNet decoder with scSE attention for edge preservation
# ============================================================================

class MixTransformerUNet(nn.Module):
    """
    Mix Transformer UNet for PV Panel Segmentation.


    """

    def __init__(
        self,
        encoder_name: str = 'mit_b4',  # Mix Transformer B4
        encoder_weights: str = 'imagenet',
        num_classes: int = Config.NUM_CLASSES,
        decoder_channels: tuple = (256, 128, 64, 32, 16)
    ):
        """
        Initialize Mix Transformer UNet.

        Args:
            encoder_name: Mix Transformer variant
            encoder_weights: Pre-trained weights from ImageNet
            num_classes: Number of output classes (1 for binary)
            decoder_channels: Channel dimensions in decoder blocks
        """
        super(MixTransformerUNet, self).__init__()

        # Build model using segmentation_models_pytorch
        self.model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=num_classes,
            decoder_channels=decoder_channels,
            decoder_use_batchnorm=True,
            decoder_attention_type='scse'  # Concurrent spatial & channel SE
        )

        self.encoder_name = encoder_name
        self.num_classes = num_classes

        print(f"✅ Model initialized: {encoder_name.upper()} + UNet + scSE Attention")
        print(f"   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Input tensor [B, 3, H, W]

        Returns:
            Output logits [B, num_classes, H, W]
        """
        return self.model(x)

    def get_encoder_name(self) -> str:
        """Return encoder name for logging."""
        return self.encoder_name


# ============================================================================
# CREATE MODEL
# ============================================================================

print("=" * 70)
print("CREATING MIX TRANSFORMER (MiT-B4) UNET MODEL")
print("=" * 70)
print("\n🏗️  Architecture Components:")
print("  • Encoder: Mix Transformer B4 (hierarchical, 4 stages)")
print("  • Decoder: UNet with progressive upsampling")
print("  • Attention: scSE (spatial + channel squeeze-excitation)")
print("  • Skip Connections: Multi-scale feature fusion")
print("=" * 70)

# Initialize model
model = MixTransformerUNet(
    encoder_name='mit_b4',
    encoder_weights='imagenet',
    num_classes=Config.NUM_CLASSES
)

# Move to GPU with mixed precision support
model = model.to(Config.DEVICE)

# ============================================================================
# MODEL STATISTICS
# ============================================================================

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"\n📊 MODEL STATISTICS:")
print(f"  {'─' * 50}")
print(f"  Total Parameters:       {total_params:,}")
print(f"  Trainable Parameters:   {trainable_params:,}")
print(f"  Non-trainable Params:   {non_trainable_params:,}")
print(f"  Model Size (FP32):      {total_params * 4 / 1e6:.2f} MB")
print(f"  Model Size (FP16):      {total_params * 2 / 1e6:.2f} MB")
print(f"  {'─' * 50}")

# ============================================================================
# FORWARD PASS TEST
# ============================================================================

print(f"\n🧪 TESTING FORWARD PASS:")
print(f"  {'─' * 50}")

# Create dummy batch
dummy_input = torch.randn(2, 3, Config.IMG_SIZE, Config.IMG_SIZE).to(Config.DEVICE)

# Test inference
with torch.no_grad():
    dummy_output = model(dummy_input)

print(f"  Input Shape:   {list(dummy_input.shape)}")
print(f"  Output Shape:  {list(dummy_output.shape)}")
print(f"  Output Range:  [{dummy_output.min():.3f}, {dummy_output.max():.3f}]")
print(f"  Memory Usage:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  {'─' * 50}")
print(f"  ✅ Forward pass successful!")

# Clean up test tensors
del dummy_input, dummy_output
torch.cuda.empty_cache()

print(f"\n{'=' * 70}")
print("🎯 TARGET PERFORMANCE:")
print(f"{'=' * 70}")
print("  Dataset  | Baseline mIoU | Target mIoU | Confidence")
print(f"  {'-' * 62}")
print("  PV01     |    91.48%     |   >92.0%    |    ✅ High")
print("  PVP      |    83.39%     |   >84.0%    |    ✅ Very High")
print("  BDAPPV   |    66.14%     |   >67.0%    |    ✅ High")
print(f"{'=' * 70}")
print("\n✅ Model ready for training!")
print("=" * 70)

CREATING MIX TRANSFORMER (MiT-B4) UNET MODEL

🏗️  Architecture Components:
  • Encoder: Mix Transformer B4 (hierarchical, 4 stages)
  • Decoder: UNet with progressive upsampling
  • Attention: scSE (spatial + channel squeeze-excitation)
  • Skip Connections: Multi-scale feature fusion
Downloading: "https://github.com/qubvel/segmentation_models.pytorch/releases/download/v0.0.2/mit_b4.pth" to /root/.cache/torch/hub/checkpoints/mit_b4.pth


100%|██████████| 234M/234M [00:07<00:00, 31.6MB/s]


✅ Model initialized: MIT_B4 + UNet + scSE Attention
   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder

📊 MODEL STATISTICS:
  ──────────────────────────────────────────────────
  Total Parameters:       64,248,664
  Trainable Parameters:   64,248,664
  Non-trainable Params:   0
  Model Size (FP32):      256.99 MB
  Model Size (FP16):      128.50 MB
  ──────────────────────────────────────────────────

🧪 TESTING FORWARD PASS:
  ──────────────────────────────────────────────────
  Input Shape:   [2, 3, 512, 512]
  Output Shape:  [2, 1, 512, 512]
  Output Range:  [-4.333, 5.165]
  Memory Usage:  0.28 GB
  ──────────────────────────────────────────────────
  ✅ Forward pass successful!

🎯 TARGET PERFORMANCE:
  Dataset  | Baseline mIoU | Target mIoU | Confidence
  --------------------------------------------------------------
  PV01     |    91.48%     |   >92.0%    |    ✅ High
  PVP      |    83.39%     |   >84.0%    |    ✅ Very High
  BDAPPV   |    66.14%     |   >67.0

## 📉 Loss Functions & Evaluation Metrics

Implementing hybrid loss and comprehensive metrics for PV panel segmentation:

### Loss Function:
- **Dice Loss** - Optimizes IoU directly, handles class imbalance
- **Binary Cross-Entropy (BCE)** - Pixel-wise classification
- **Combined Loss** - Weighted sum for balanced optimization

### Evaluation Metrics:
- **mIoU (Mean Intersection over Union)** - Primary metric for baselines
- **F1-Score (Dice Score)** - Harmonic mean of precision & recall
- **Precision** - Correct PV predictions / All PV predictions
- **Recall** - Correct PV predictions / All ground truth PV pixels



In [12]:
# ============================================================================
# LOSS FUNCTIONS & EVALUATION METRICS
# ============================================================================
# Implementing hybrid loss and comprehensive metrics for segmentation.
# ============================================================================

class DiceLoss(nn.Module):
    """
    Dice Loss for binary segmentation.
    Directly optimizes IoU metric.
    """
    def __init__(self, smooth: float = 1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Calculate Dice Loss.

        Args:
            pred: Predictions [B, C, H, W] (logits)
            target: Ground truth [B, C, H, W]

        Returns:
            Dice loss value
        """
        # Apply sigmoid to convert logits to probabilities
        pred = torch.sigmoid(pred)

        # Ensure target has a channel dimension if it's missing (e.g., [B, H, W] -> [B, 1, H, W])
        if target.ndim == pred.ndim - 1:
            target = target.unsqueeze(1)

        # Ensure both tensors have the same shape
        assert pred.shape == target.shape, f"Shape mismatch: pred {pred.shape} vs target {target.shape}"

        # Flatten spatial dimensions: [B, C, H, W] -> [B*C*H*W]
        pred_flat = pred.contiguous().view(-1)
        target_flat = target.contiguous().view(-1)

        # Calculate Dice coefficient
        intersection = (pred_flat * target_flat).sum()
        dice = (2. * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )

        return 1 - dice


class CombinedLoss(nn.Module):
    """
    Combined Dice + BCE Loss for better edge preservation.
    """
    def __init__(self, dice_weight: float = 0.5, bce_weight: float = 0.5):
        super(CombinedLoss, self).__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Calculate combined loss."""
        # Ensure target has a channel dimension for BCE loss if pred has one and target doesn't
        # This makes the target shape [B, 1, H, W] if pred is [B, 1, H, W] and target is [B, H, W]
        if pred.ndim == target.ndim + 1 and pred.shape[1] == 1:
            bce_target = target.unsqueeze(1)
        else:
            bce_target = target

        dice = self.dice_loss(pred, target)
        bce = self.bce_loss(pred, bce_target)
        return self.dice_weight * dice + self.bce_weight * bce


class SegmentationMetrics:
    """
    Calculate comprehensive segmentation metrics.
    """
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        """
        Reset TP, FP, FN, TN counts.
        """
        self.tp = 0
        self.fp = 0
        self.fn = 0
        self.tn = 0

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """
        Update metrics with batch predictions.

        Args:
            pred: Predictions [B, C, H, W] (logits)
            target: Ground truth [B, C, H, W]
        """
        # Convert logits to binary predictions
        pred = (torch.sigmoid(pred) > self.threshold).float()

        # Ensure target has a channel dimension if it's missing (e.g., [B, H, W] -> [B, 1, H, W])
        if target.ndim == pred.ndim - 1:
            target = target.unsqueeze(1)

        # Ensure same shape
        assert pred.shape == target.shape, f"Shape mismatch: pred {pred.shape} vs target {target.shape}"

        # Calculate confusion matrix components
        self.tp += ((pred == 1) & (target == 1)).sum().item()
        self.fp += ((pred == 1) & (target == 0)).sum().item()
        self.fn += ((pred == 0) & (target == 1)).sum().item()
        self.tn += ((pred == 0) & (target == 0)).sum().item()

    def get_metrics(self) -> Dict[str, float]:
        """Calculate all metrics."""
        epsilon = 1e-7

        # IoU (Intersection over Union)
        iou = self.tp / (self.tp + self.fp + self.fn + epsilon)

        # Precision & Recall
        precision = self.tp / (self.tp + self.fp + epsilon)
        recall = self.tp / (self.tp + self.fn + epsilon)

        # F1-Score (Dice Score)
        f1 = 2 * precision * recall / (precision + recall + epsilon)

        # Accuracy
        accuracy = (self.tp + self.tn) / (self.tp + self.fp + self.fn + self.tn + epsilon)

        return {
            'mIoU': iou * 100,  # Convert to percentage
            'F1-Score': f1 * 100,
            'Precision': precision * 100,
            'Recall': recall * 100,
            'Accuracy': accuracy * 100
        }


# ============================================================================
# INITIALIZE LOSS & METRICS
# ============================================================================

print("=" * 70)
print("INITIALIZING LOSS FUNCTIONS & METRICS (FIXED)")
print("=" * 70)

# Create loss function
criterion = CombinedLoss(dice_weight=1.0, bce_weight=0.0) # ablation A2
criterion = criterion.to(Config.DEVICE)

print("\n✅ Loss Function: Combined Dice (100%) + BCE (0%)")
print("   - Dice Loss: Optimizes IoU directly")
print("   - BCE Loss: Pixel-wise classification")
print("   - Weighted combination for balanced training")
print("   - Fixed tensor shape handling")

print("\n✅ Metrics Initialized:")
print("   - mIoU (Mean IoU) - Primary metric for baseline comparison")
print("   - F1-Score (Dice) - Harmonic mean of precision & recall")
print("   - Precision - Correct predictions / Total predictions")
print("   - Recall - Correct predictions / Total ground truth")
print("   - Accuracy - Overall pixel accuracy")

print("\n🎯 Baseline Targets:")
print("   PV01:    mIoU > 92.0% (vs 91.48% baseline)")
print("   PVP:     mIoU > 84.0% (vs 83.39% baseline)")
print("   BDAPPV:  mIoU > 67.0% (vs 66.14% baseline)")

print("=" * 70)


INITIALIZING LOSS FUNCTIONS & METRICS (FIXED)

✅ Loss Function: Combined Dice (100%) + BCE (0%)
   - Dice Loss: Optimizes IoU directly
   - BCE Loss: Pixel-wise classification
   - Weighted combination for balanced training
   - Fixed tensor shape handling

✅ Metrics Initialized:
   - mIoU (Mean IoU) - Primary metric for baseline comparison
   - F1-Score (Dice) - Harmonic mean of precision & recall
   - Precision - Correct predictions / Total predictions
   - Recall - Correct predictions / Total ground truth
   - Accuracy - Overall pixel accuracy

🎯 Baseline Targets:
   PV01:    mIoU > 92.0% (vs 91.48% baseline)
   PVP:     mIoU > 84.0% (vs 83.39% baseline)
   BDAPPV:  mIoU > 67.0% (vs 66.14% baseline)


## ⚙️ Optimizer & Learning Rate Scheduler

Configuring training optimization strategy:

### Optimizer:
- **AdamW** - Adam with weight decay for better generalization
- Learning rate: 1e-4 (stable for transformer fine-tuning)
- Weight decay: 1e-4 (L2 regularization)
- Betas: (0.9, 0.999) - standard momentum parameters

### Learning Rate Scheduler:
- **Cosine Annealing with Warm Restarts** - Smooth LR decay with periodic restarts
- Improves convergence and helps escape local minima
- Warm restarts prevent premature convergence

### Mixed Precision Training:
- **AMP (Automatic Mixed Precision)** - FP16 computation for 2-3x speedup
- Maintains FP32 master weights for stability
- Reduces memory usage by ~40%

In [13]:
# ============================================================================
# OPTIMIZER & LEARNING RATE SCHEDULER
# ============================================================================
# Configure AdamW optimizer with cosine annealing for stable training.
# ============================================================================

from torch.cuda.amp import GradScaler, autocast

# ============================================================================
# OPTIMIZER SETUP
# ============================================================================

print("=" * 70)
print("CONFIGURING OPTIMIZER & SCHEDULER")
print("=" * 70)

# AdamW optimizer with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

print(f"\n✅ Optimizer: AdamW")
print(f"   - Learning Rate: {Config.LEARNING_RATE}")
print(f"   - Weight Decay: {Config.WEIGHT_DECAY}")
print(f"   - Betas: (0.9, 0.999)")

# ============================================================================
# LEARNING RATE SCHEDULER
# ============================================================================

# Cosine annealing with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,  # Restart every 10 epochs
    T_mult=2,  # Double the restart interval after each restart
    eta_min=1e-6  # Minimum learning rate
)

print(f"\n✅ Scheduler: CosineAnnealingWarmRestarts")
print(f"   - Initial restart period: 10 epochs")
print(f"   - Period multiplier: 2x after each restart")
print(f"   - Minimum LR: 1e-6")

# ============================================================================
# MIXED PRECISION TRAINING SETUP
# ============================================================================

# GradScaler for automatic mixed precision
scaler = GradScaler()

print(f"\n✅ Mixed Precision Training (AMP): Enabled")
print(f"   - FP16 computation for 2-3x speedup")
print(f"   - FP32 master weights for stability")
print(f"   - Automatic gradient scaling")

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

print(f"\n{'=' * 70}")
print("TRAINING CONFIGURATION")
print(f"{'=' * 70}")

print(f"\n📊 Hyperparameters:")
print(f"   - Batch Size: {Config.BATCH_SIZE}")
print(f"   - Image Size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"   - Max Epochs: {Config.EPOCHS}")
print(f"   - Learning Rate: {Config.LEARNING_RATE}")
print(f"   - Weight Decay: {Config.WEIGHT_DECAY}")

print(f"\n🎯 Training Strategy:")
print(f"   - Loss: Combined Dice (50%) + BCE (50%)")
print(f"   - Optimizer: AdamW with cosine annealing")
print(f"   - Precision: Mixed FP16/FP32")
print(f"   - Device: {Config.DEVICE}")

print(f"\n💾 Checkpointing:")
print(f"   - Save best model based on validation mIoU")
print(f"   - Early stopping patience: 15 epochs")

print("=" * 70)
print("✅ Training setup complete! Ready to train!")
print("=" * 70)

CONFIGURING OPTIMIZER & SCHEDULER

✅ Optimizer: AdamW
   - Learning Rate: 0.0001
   - Weight Decay: 1e-05
   - Betas: (0.9, 0.999)

✅ Scheduler: CosineAnnealingWarmRestarts
   - Initial restart period: 10 epochs
   - Period multiplier: 2x after each restart
   - Minimum LR: 1e-6

✅ Mixed Precision Training (AMP): Enabled
   - FP16 computation for 2-3x speedup
   - FP32 master weights for stability
   - Automatic gradient scaling

TRAINING CONFIGURATION

📊 Hyperparameters:
   - Batch Size: 8
   - Image Size: 512x512
   - Max Epochs: 150
   - Learning Rate: 0.0001
   - Weight Decay: 1e-05

🎯 Training Strategy:
   - Loss: Combined Dice (50%) + BCE (50%)
   - Optimizer: AdamW with cosine annealing
   - Precision: Mixed FP16/FP32
   - Device: cuda

💾 Checkpointing:
   - Save best model based on validation mIoU
   - Early stopping patience: 15 epochs
✅ Training setup complete! Ready to train!


## 🔄 Training & Validation Loop

Implementing comprehensive training pipeline with:

### Training Loop:
- **Batch Processing** - Forward pass through model
- **Mixed Precision** - FP16 computation with gradient scaling
- **Loss Computation** - Combined Dice + BCE
- **Backpropagation** - Gradient computation and weight updates
- **Progress Tracking** - Real-time metrics with tqdm

### Validation Loop:
- **Evaluation** - Model performance on validation set
- **Metrics Calculation** - mIoU, F1-Score, Precision, Recall
- **Model Checkpointing** - Save best weights based on mIoU
- **Early Stopping** - Prevent overfitting with patience mechanism

### Training Strategy:
We'll train **separate models** for each dataset to ensure optimal performance:
- **PVP Model** - Trained on PVP dataset
- **BDAPPV Model** - Trained on BDAPPV dataset
- **PV01 Model** - Trained on PV01 dataset

This approach allows dataset-specific optimization and fair comparison with baselines.

In [14]:
from tqdm.auto import tqdm
import time

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: str,
    epoch: int
) -> Dict[str, float]:
    """
    Train model for one epoch.

    Args:
        model: Neural network model
        dataloader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        scaler: Gradient scaler for mixed precision
        device: Device to train on
        epoch: Current epoch number

    Returns:
        Dictionary with average loss and metrics
    """
    model.train()

    running_loss = 0.0
    metrics = SegmentationMetrics()

    # Clear CUDA cache at the beginning of each epoch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Progress bar
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")

    for batch_idx, batch in enumerate(pbar):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, masks)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update metrics
        running_loss += loss.item()
        metrics.update(outputs.detach(), masks.detach())

        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Calculate average metrics
    avg_loss = running_loss / len(dataloader)
    epoch_metrics = metrics.get_metrics()
    epoch_metrics['loss'] = avg_loss

    return epoch_metrics

def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: str,
    epoch: int
) -> Dict[str, float]:
    """
    Validate model on validation set.

    Args:
        model: Neural network model
        dataloader: Validation data loader
        criterion: Loss function
        device: Device to validate on
        epoch: Current epoch number

    Returns:
        Dictionary with validation metrics
    """
    model.eval()

    running_loss = 0.0
    metrics = SegmentationMetrics()

    # Clear CUDA cache at the beginning of validation
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Progress bar
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]  ")

    with torch.no_grad():
        for batch in pbar:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            # Forward pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)

            # Update metrics
            running_loss += loss.item()
            metrics.update(outputs, masks)

            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Calculate average metrics
    avg_loss = running_loss / len(dataloader)
    val_metrics = metrics.get_metrics()
    val_metrics['loss'] = avg_loss

    return val_metrics

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: GradScaler,
    device: str,
    num_epochs: int,
    dataset_name: str,
    patience: int = 15
) -> Dict[str, list]:
    """
    Complete training loop with checkpointing and early stopping.

    Args:
        model: Neural network model
        train_loader: Training data loader
        val_loader: Validation data loader
        criterion: Loss function
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        scaler: Gradient scaler
        device: Device to train on
        num_epochs: Maximum number of epochs
        dataset_name: Name of dataset (for checkpoint naming)
        patience: Early stopping patience

    Returns:
        Dictionary with training history
    """
    print(f"\n{'=' * 70}")
    print(f"TRAINING MODEL ON {dataset_name} DATASET")
    print(f"{'=' * 70}\n")

    # Training history
    history = {
        'train_loss': [],
        'train_miou': [],
        'train_f1': [],
        'val_loss': [],
        'val_miou': [],
        'val_f1': [],
        'lr': []
    }

    # Best model tracking
    best_miou = 0.0
    patience_counter = 0
    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()

        # Get current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        # Training
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, epoch
        )

        # Validation
        val_metrics = validate(model, val_loader, criterion, device, epoch)

        # Update scheduler
        scheduler.step()

        # Save history
        history['train_loss'].append(train_metrics['loss'])
        history['train_miou'].append(train_metrics['mIoU'])
        history['train_f1'].append(train_metrics['F1-Score'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_miou'].append(val_metrics['mIoU'])
        history['val_f1'].append(val_metrics['F1-Score'])
        history['lr'].append(current_lr)

        # Epoch summary
        epoch_time = time.time() - epoch_start
        print(f"\n{'─' * 70}")
        print(f"Epoch {epoch}/{num_epochs} Summary:")
        print(f"  Train Loss: {train_metrics['loss']:.4f} | mIoU: {train_metrics['mIoU']:.2f}% | F1: {train_metrics['F1-Score']:.2f}%")
        print(f"  Val   Loss: {val_metrics['loss']:.4f} | mIoU: {val_metrics['mIoU']:.2f}% | F1: {val_metrics['F1-Score']:.2f}%")
        print(f"  Learning Rate: {current_lr:.6f} | Time: {epoch_time:.1f}s")

        # Save best model
        if val_metrics['mIoU'] > best_miou:
            best_miou = val_metrics['mIoU']
            patience_counter = 0

            # Save checkpoint
            checkpoint_path = f'best_model_{dataset_name}.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_miou': best_miou,
                'history': history
            }, checkpoint_path)

            print(f"  ✅ New best mIoU: {best_miou:.2f}% - Model saved!")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")

        print(f"{'─' * 70}\n")

        # Early stopping
        if patience_counter >= patience:
            print(f"☢️  Early stopping triggered after {epoch} epochs")
            break

    # Training complete
    total_time = time.time() - start_time
    print(f"\n{'=' * 70}")
    print(f"TRAINING COMPLETE!")
    print(f"  Total Time: {total_time / 60:.1f} minutes")
    print(f"  Best Validation mIoU: {best_miou:.2f}%")
    print(f"  Model saved: best_model_{dataset_name}.pth")
    print(f"{'=' * 70}\n")

    return history


print("=" * 70)
print("✅ Training & Validation Functions Defined")
print("=" * 70)
print("\nFunctions available:")
print("  • train_one_epoch() - Single epoch training with mixed precision")
print("  • validate() - Validation with metrics calculation")
print("  • train_model() - Complete training loop with checkpointing")
print("\nReady to start training on all 3 datasets!")
print("=" * 70)

✅ Training & Validation Functions Defined

Functions available:
  • train_one_epoch() - Single epoch training with mixed precision
  • validate() - Validation with metrics calculation
  • train_model() - Complete training loop with checkpointing

Ready to start training on all 3 datasets!


## 🚀 Execute Training on All Datasets

Each model will be saved independently as `best_model_{dataset_name}.pth` when it achieves the best validation mIoU. Training will automatically stop early if no improvement is seen for 15 consecutive epochs.


In [15]:
# ============================================================================
# TRAIN MODELS ON ALL THREE DATASETS
# ============================================================================


import warnings
warnings.filterwarnings('ignore')

# Store all training histories
all_histories = {}

# ============================================================================
# TRAIN ON EACH DATASET
# ============================================================================

datasets_to_train = ['PVP', 'BDAPPV', 'PV01']

for dataset_name in datasets_to_train:
    print(f"\n\n{'#' * 70}")
    print(f"{'#' * 70}")
    print(f"###  STARTING TRAINING: {dataset_name} DATASET")
    print(f"{'#' * 70}")
    print(f"{'#' * 70}\n")

    # Clear CUDA cache before initializing model for a new dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Create fresh model for this dataset
    model = MixTransformerUNet(
        encoder_name='mit_b4',
        encoder_weights='imagenet',
        num_classes=Config.NUM_CLASSES
    )
    model = model.to(Config.DEVICE)

    # Create fresh optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=2,
        eta_min=1e-6
    )

    scaler = GradScaler()

    # Get dataloaders for this dataset
    train_loader = all_dataloaders[dataset_name]['train']
    val_loader = all_dataloaders[dataset_name]['val']

    print(f"Dataset Info:")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")

    # Train model
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=Config.DEVICE,
        num_epochs=Config.EPOCHS,
        dataset_name=dataset_name,
        patience=15
    )

    # Store history
    all_histories[dataset_name] = history

    # Clear memory
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    print(f"\n✅ {dataset_name} training complete!\n")

# ============================================================================
# TRAINING SUMMARY
# ============================================================================

print(f"\n\n{'=' * 70}")
print("✨ ALL TRAINING COMPLETE!")
print(f"{'=' * 70}\n")

print("📊 Best Validation mIoU Results:")
print(f"{'─' * 70}")

for dataset_name in datasets_to_train:
    best_miou = max(all_histories[dataset_name]['val_miou'])
    best_epoch = all_histories[dataset_name]['val_miou'].index(best_miou) + 1

    # Get baseline for comparison
    baselines = {
        'PV01': 91.48,
        'PVP': 83.39,
        'BDAPPV': 66.14
    }

    baseline = baselines[dataset_name]
    improvement = best_miou - baseline
    status = "✅ BEAT BASELINE" if improvement > 0 else "⚠️ Below baseline"

    print(f"\n{dataset_name}:")
    print(f"  Best Val mIoU: {best_miou:.2f}% (Epoch {best_epoch})")
    print(f"  Baseline:      {baseline:.2f}%")
    print(f"  Improvement:   {improvement:+.2f}%")
    print(f"  Status:        {status}")

print(f"\n{'=' * 70}")
print("Models saved:")
for dataset_name in datasets_to_train:
    print(f"  • best_model_{dataset_name}.pth")
print(f"{'=' * 70}\n")



######################################################################
######################################################################
###  STARTING TRAINING: PVP DATASET
######################################################################
######################################################################

✅ Model initialized: MIT_B4 + UNet + scSE Attention
   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder
Dataset Info:
  Train batches: 406
  Val batches: 87

TRAINING MODEL ON PVP DATASET



Epoch 1 [Train]:   0%|          | 0/406 [00:00<?, ?it/s]

Epoch 1 [Val]  :   0%|          | 0/87 [00:00<?, ?it/s]


──────────────────────────────────────────────────────────────────────
Epoch 1/150 Summary:
  Train Loss: 0.2087 | mIoU: 77.02% | F1: 87.02%
  Val   Loss: 0.0944 | mIoU: 87.46% | F1: 93.31%
  Learning Rate: 0.000100 | Time: 2502.0s
  ✅ New best mIoU: 87.46% - Model saved!
──────────────────────────────────────────────────────────────────────



Epoch 2 [Train]:   0%|          | 0/406 [00:00<?, ?it/s]

...

Epoch 39 [Val]  :   0%|          | 0/12 [00:00<?, ?it/s]


──────────────────────────────────────────────────────────────────────
Epoch 39/150 Summary:
  Train Loss: 0.0416 | mIoU: 93.98% | F1: 96.89%
  Val   Loss: 0.0318 | mIoU: 94.91% | F1: 97.39%
  Learning Rate: 0.000091 | Time: 36.4s
  No improvement (14/15)
──────────────────────────────────────────────────────────────────────



Epoch 40 [Train]:   0%|          | 0/56 [00:00<?, ?it/s]

Epoch 40 [Val]  :   0%|          | 0/12 [00:00<?, ?it/s]


──────────────────────────────────────────────────────────────────────
Epoch 40/150 Summary:
  Train Loss: 0.0382 | mIoU: 94.19% | F1: 97.01%
  Val   Loss: 0.0310 | mIoU: 95.01% | F1: 97.44%
  Learning Rate: 0.000088 | Time: 36.3s
  No improvement (15/15)
──────────────────────────────────────────────────────────────────────

☢️  Early stopping triggered after 40 epochs

TRAINING COMPLETE!
  Total Time: 38.1 minutes
  Best Validation mIoU: 95.15%
  Model saved: best_model_PV01.pth


✅ PV01 training complete!



✨ ALL TRAINING COMPLETE!

📊 Best Validation mIoU Results:
──────────────────────────────────────────────────────────────────────

PVP:
  Best Val mIoU: 92.35% (Epoch 29)
  Baseline:      83.39%
  Improvement:   +8.96%
  Status:        ✅ BEAT BASELINE

BDAPPV:
  Best Val mIoU: 73.75% (Epoch 30)
  Baseline:      66.14%
  Improvement:   +7.61%
  Status:        ✅ BEAT BASELINE

PV01:
  Best Val mIoU: 95.15% (Epoch 25)
  Baseline:      91.48%
  Improvement:   +3.67%
  Status:       

In [ ]:
# ============================================================================
# SAVE ALL MODELS TO GOOGLE DRIVE
# ============================================================================

import torch
import os
import shutil

# Models are currently in /content/ directory
models_to_save = ['best_model_PVP.pth', 'best_model_BDAPPV.pth', 'best_model_PV01.pth']

# Destination in Google Drive
drive_models_dir = '/content/drive/MyDrive/TransformerUNet_PV_Segmentation/models'

print("Copying models to Google Drive...")
print("=" * 70)

for model_name in models_to_save:
    src = f'/content/{model_name}'
    dst = os.path.join(drive_models_dir, model_name)

    if os.path.exists(src):
        shutil.copy(src, dst)

        # Verify
        checkpoint = torch.load(dst)
        print(f"✅ {model_name}")
        print(f"   Best mIoU: {checkpoint['best_miou']:.2f}%")
        print(f"   Saved to: {dst}")
        print()
    else:
        print(f"❌ {model_name} not found in /content/")

print("=" * 70)
print("✅ All models safely saved to Google Drive!")

Copying models to Google Drive...
✅ best_model_PVP.pth
   Best mIoU: 92.29%
   Saved to: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/models/best_model_PVP.pth

✅ best_model_BDAPPV.pth
   Best mIoU: 73.81%
   Saved to: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/models/best_model_BDAPPV.pth

✅ best_model_PV01.pth
   Best mIoU: 95.26%
   Saved to: /content/drive/MyDrive/TransformerUNet_PV_Segmentation/models/best_model_PV01.pth

✅ All models safely saved to Google Drive!


In [18]:
# ============================================================================
# STEP 1: LOAD ALL TRAINED MODELS FOR TEST EVALUATION
# ============================================================================

import torch
import torch.nn as nn
from collections import OrderedDict

# Model paths
model_paths = {
    'PVP': '/content/best_model_PVP.pth',
    'PV01': '/content/best_model_PV01.pth',
    'BDAPPV': '/content/best_model_BDAPPV.pth'
}


# Load models
loaded_models = {}

for dataset_name, model_path in model_paths.items():
    # Create model architecture
    model = MixTransformerUNet(
        encoder_name='mit_b4',
        encoder_weights=None,  # Don't load ImageNet weights
        num_classes=Config.NUM_CLASSES
    )

    # Load trained weights
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(Config.DEVICE)
    model.eval()  # Set to evaluation mode

    loaded_models[dataset_name] = model

    print(f"✅ {dataset_name}: Loaded model with mIoU {checkpoint['best_miou']:.2f}%")

print("\n" + "=" * 70)
print("✅ All 3 models loaded successfully!")
print("=" * 70)

✅ Model initialized: MIT_B4 + UNet + scSE Attention
   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder
✅ PVP: Loaded model with mIoU 92.35%
✅ Model initialized: MIT_B4 + UNet + scSE Attention
   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder
✅ PV01: Loaded model with mIoU 95.15%
✅ Model initialized: MIT_B4 + UNet + scSE Attention
   Architecture: Hierarchical Transformer Encoder + Multi-scale Decoder
✅ BDAPPV: Loaded model with mIoU 73.75%

✅ All 3 models loaded successfully!


In [19]:
# ============================================================================
# STEP 1: TEST SET EVALUATION FUNCTION
# ============================================================================

import torch
from tqdm import tqdm
import numpy as np

def evaluate_on_test_set(model, test_loader, device='cuda'):
    """
    Evaluate model on test set and return comprehensive metrics.
    """
    model.eval()

    # Initialize metrics
    all_ious = []
    all_f1s = []
    all_precisions = []
    all_recalls = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            # Forward pass
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).float()

            # Calculate metrics for each image in batch
            for pred, mask in zip(preds, masks):
                pred = pred.cpu().numpy()
                mask = mask.cpu().numpy()

                # Flatten
                pred_flat = pred.flatten()
                mask_flat = mask.flatten()

                # Calculate IoU
                intersection = np.logical_and(pred_flat, mask_flat).sum()
                union = np.logical_or(pred_flat, mask_flat).sum()
                iou = intersection / (union + 1e-8)
                all_ious.append(iou)

                # Calculate Precision, Recall, F1
                tp = intersection
                fp = (pred_flat == 1).sum() - tp
                fn = (mask_flat == 1).sum() - tp

                precision = tp / (tp + fp + 1e-8)
                recall = tp / (tp + fn + 1e-8)
                f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

                all_precisions.append(precision)
                all_recalls.append(recall)
                all_f1s.append(f1)

    # Calculate mean metrics
    results = {
        'mIoU': np.mean(all_ious) * 100,
        'F1': np.mean(all_f1s) * 100,
        'Precision': np.mean(all_precisions) * 100,
        'Recall': np.mean(all_recalls) * 100,
        'std_IoU': np.std(all_ious) * 100,  # For statistical testing
        'all_ious': all_ious  # For paired t-test
    }

    return results

print("✅ Test evaluation function created!")

✅ Test evaluation function created!


In [20]:
# ============================================================================
# STEP 1: EVALUATE ALL MODELS ON TEST SETS
# ============================================================================

# Store all test results
test_results = {}

# Baseline
baseline_results = {
    'PVP': {'mIoU': 83.39},
    'PV01': {'mIoU': 91.48},
    'BDAPPV': {'mIoU': 66.14}
}

print("\n" + "=" * 80)
print("🧪 STARTING TEST SET EVALUATION")
print("=" * 80)

for dataset_name in ['PVP', 'PV01', 'BDAPPV']:
    print(f"\n📊 Evaluating {dataset_name} on TEST SET...")

    # Get test dataloader
    test_loader = all_dataloaders[dataset_name]['test']

    # Evaluate
    results = evaluate_on_test_set(loaded_models[dataset_name], test_loader, Config.DEVICE)

    # Store results
    test_results[dataset_name] = results

    # Print results
    baseline_miou = baseline_results[dataset_name]['mIoU']
    improvement = results['mIoU'] - baseline_miou

    print(f"\n{'=' * 70}")
    print(f"📈 {dataset_name} TEST RESULTS:")
    print(f"{'=' * 70}")
    print(f"  mIoU:      {results['mIoU']:.2f}% (Baseline: {baseline_miou:.2f}% | Δ: {improvement:+.2f}%)")
    print(f"  F1-Score:  {results['F1']:.2f}%")
    print(f"  Precision: {results['Precision']:.2f}%")
    print(f"  Recall:    {results['Recall']:.2f}%")
    print(f"  Std IoU:   {results['std_IoU']:.2f}%")
    print(f"{'=' * 70}")

print("\n" + "=" * 80)
print("✅ TEST SET EVALUATION COMPLETE!")
print("=" * 80)


🧪 STARTING TEST SET EVALUATION

📊 Evaluating PVP on TEST SET...


Testing: 100%|██████████| 87/87 [08:47<00:00,  6.06s/it]



📈 PVP TEST RESULTS:
  mIoU:      90.84% (Baseline: 83.39% | Δ: +7.45%)
  F1-Score:  94.72%
  Precision: 94.55%
  Recall:    95.04%
  Std IoU:   9.95%

📊 Evaluating PV01 on TEST SET...


Testing: 100%|██████████| 13/13 [00:47<00:00,  3.66s/it]



📈 PV01 TEST RESULTS:
  mIoU:      89.09% (Baseline: 91.48% | Δ: -2.39%)
  F1-Score:  93.34%
  Precision: 95.11%
  Recall:    93.21%
  Std IoU:   14.50%

📊 Evaluating BDAPPV on TEST SET...


Testing: 100%|██████████| 145/145 [10:39<00:00,  4.41s/it]


📈 BDAPPV TEST RESULTS:
  mIoU:      76.03% (Baseline: 66.14% | Δ: +9.89%)
  F1-Score:  85.05%
  Precision: 85.22%
  Recall:    87.22%
  Std IoU:   16.58%

✅ TEST SET EVALUATION COMPLETE!
